In [15]:
import pandas as pd
from google.colab import files
uploaded = files.upload()
# =========================
# LOAD DATASET
# =========================
# TODO: Change this to the correct file path.
# Make sure the file exists in your Colab environment or provide the full path.

df = pd.read_csv(list(uploaded.keys())[0])
df.rename(columns={'tick': 'Ticker'}, inplace=True)


Saving Trading View (Mission 300cr) - 2025.csv to Trading View (Mission 300cr) - 2025 (3).csv


In [16]:
print("="*60)
print("FIRST 5 ROWS")
print("="*60)
print(df.head())

print("\n" + "="*60)
print("COLUMN NAMES")
print("="*60)
print(df.columns.tolist())

# =========================
# BASIC CLEANING
# =========================
if 'Date' in df.columns:
    df['Date'] = pd.to_datetime(
        df['Date'].astype(int),
        unit='D',
        origin='1899-12-30'
    )
# remove fully empty rows
df = df.dropna(how='all')

# =========================
# OVERALL SHAPE
# =========================
print("\n" + "="*60)
print("DATASET SHAPE")
print("="*60)
print(f"Rows: {df.shape[0]}")
print(f"Columns: {df.shape[1]}")

# =========================
# DATE RANGE
# =========================
if 'Date' in df.columns:
    print("\n" + "="*60)
    print("DATE RANGE")
    print("="*60)
    print("Start:", df['Date'].min())
    print("End  :", df['Date'].max())
    print("Total Days Covered:", (df['Date'].max() - df['Date'].min()).days)

# =========================
# MISSING VALUES
# =========================
print("\n" + "="*60)
print("MISSING VALUES")
print("="*60)
print(df.isnull().sum())

# =========================
# DUPLICATES
# =========================
print("\n" + "="*60)
print("DUPLICATES")
print("="*60)
print("Exact duplicate rows:", df.duplicated().sum())

if {'Ticker', 'Date'}.issubset(df.columns):
    ticker_date_dupes = df.duplicated(subset=['Ticker', 'Date']).sum()
    print("Duplicate Ticker+Date rows:", ticker_date_dupes)

# =========================
# TICKER ANALYSIS
# =========================
if 'Ticker' in df.columns:
    print("\n" + "="*60)
    print("UNIQUE TICKERS")
    print("="*60)
    tickers = df['Ticker'].unique()
    print(tickers)
    print("Total unique tickers:", len(tickers))

    print("\n" + "="*60)
    print("ROWS PER TICKER")
    print("="*60)
    print(df['Ticker'].value_counts())

# =========================
# NUMERIC COLUMN CHECK
# =========================
numeric_cols = ['Open', 'High', 'Low', 'Close', 'Volume']

for col in numeric_cols:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors='coerce')

print("\n" + "="*60)
print("NUMERIC SUMMARY")
print("="*60)
print(df.describe())

# =========================
# MARKET SPLIT (INDIA vs US rough heuristic)
# =========================
if 'Ticker' in df.columns:
    indian_keywords = ['NIFTY', 'SENSEX', 'RELIANCE', 'TCS', 'SBIN', '.NS', '.BO']

    def classify_market(ticker):
        ticker = str(ticker).upper()
        for k in indian_keywords:
            if k in ticker:
                return 'India'
        return 'US/Other'

    df['Market'] = df['Ticker'].apply(classify_market)

    print("\n" + "="*60)
    print("MARKET SPLIT")
    print("="*60)
    print(df['Market'].value_counts())

# =========================
# DAILY GAP CHECK
# =========================
if {'Ticker', 'Date'}.issubset(df.columns):
    print("\n" + "="*60)
    print("DATE GAP CHECK PER TICKER")
    print("="*60)

    for ticker in df['Ticker'].unique():
        temp = df[df['Ticker'] == ticker].sort_values('Date')
        gaps = temp['Date'].diff().dt.days
        print(f"\n{ticker}")
        print("Max gap:", gaps.max())
        print("Avg gap:", round(gaps.mean(), 2))

# =========================
# OHLC VALIDATION
# =========================
if set(['Open', 'High', 'Low', 'Close']).issubset(df.columns):
    invalid_rows = df[
        (df['High'] < df['Low']) |
        (df['High'] < df['Open']) |
        (df['High'] < df['Close']) |
        (df['Low'] > df['Open']) |
        (df['Low'] > df['Close'])
    ]

    print("\n" + "="*60)
    print("OHLC VALIDATION")
    print("="*60)
    print("Invalid OHLC rows:", len(invalid_rows))

print("\n" + "="*60)
print("AUDIT COMPLETE")
print("="*60)

FIRST 5 ROWS
          Date   Open   High    Low  Close   Volume Ticker
0  45293.66667  48.32  48.47  46.19  46.85  5049423   NFLX
1  45294.66667  46.73  47.51  46.58  47.03  3443694   NFLX
2  45295.66667  47.30  48.07  46.65  47.47  3636513   NFLX
3  45296.66667  47.65  47.95  47.18  47.41  2631250   NFLX
4  45299.66667  47.39  48.52  47.37  48.50  3675775   NFLX

COLUMN NAMES
['Date', 'Open', 'High', 'Low', 'Close', 'Volume', 'Ticker']

DATASET SHAPE
Rows: 4708
Columns: 7

DATE RANGE
Start: 2024-01-01 00:00:00
End  : 2026-05-18 00:00:00
Total Days Covered: 868

MISSING VALUES
Date      0
Open      0
High      0
Low       0
Close     0
Volume    0
Ticker    0
dtype: int64

DUPLICATES
Exact duplicate rows: 0
Duplicate Ticker+Date rows: 0

UNIQUE TICKERS
['NFLX' 'RELIANCE' 'TCS' 'SBIN' 'SENSEX' 'NIFTY' 'NVIDIA' 'AAPL']
Total unique tickers: 8

ROWS PER TICKER
Ticker
NFLX        595
TCS         595
NVIDIA      595
AAPL        594
SBIN        583
RELIANCE    583
NIFTY       582
SENSEX    

In [18]:
print(invalid_rows)
df = df.drop(invalid_rows.index)

           Date      Open      High       Low     Close  Volume  Ticker Market
1376 2024-10-15     41.06     41.20     40.66     41.57      38     TCS  India
1385 2024-10-28     41.69     42.11     41.44     40.45    5702     TCS  India
2581 2024-12-03  79743.87  80949.10  80244.78  80845.75       0  SENSEX  India
2782 2025-09-24  82147.37  82045.47  81607.84  81715.63       0  SENSEX  India


In [19]:
print("Date Range:", df['Date'].min(), "to", df['Date'].max())
print("Unique Tickers:", df['Ticker'].nunique())
print(df['Ticker'].unique())
print(df['Ticker'].value_counts())
print(df['Market'].value_counts())


Date Range: 2024-01-01 00:00:00 to 2026-05-18 00:00:00
Unique Tickers: 8
['NFLX' 'RELIANCE' 'TCS' 'SBIN' 'SENSEX' 'NIFTY' 'NVIDIA' 'AAPL']
Ticker
NFLX        595
NVIDIA      595
AAPL        594
TCS         593
SBIN        583
RELIANCE    583
NIFTY       582
SENSEX      579
Name: count, dtype: int64
Market
India       2920
US/Other    1784
Name: count, dtype: int64
